# QR and SVD Intuition

This lesson explores how QR yields an orthonormal basis with a triangular factor, and how SVD stretches space along singular directions while reconstructing the matrix via U Σ V^T.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from linalg_with_python.checks import is_orthonormal, is_upper_triangular
from linalg_with_python.decompositions import qr_gram_schmidt

FIGURES_DIR = Path.cwd() / "assets" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## QR factorization: orthonormal basis + triangular factor

We factor a tall matrix into Q (orthonormal columns) and R (upper triangular). The figure below shows how the columns of Q span the space while R encodes the combination weights.


In [ ]:

A = np.array([[1.0, 1.0], [0.5, 2.0], [0.0, 1.0]], dtype=np.float64)
qr = qr_gram_schmidt(A, method="modified")
print("Is Q orthonormal?", is_orthonormal(qr.Q))
print("Is R upper triangular?", is_upper_triangular(qr.R))

fig, ax = plt.subplots(figsize=(5, 5))
for idx, col in enumerate(A.T):
    ax.arrow(0.0, 0.0, col[0], col[1], head_width=0.05, length_includes_head=True, label=f"A col {idx+1}")
for idx, vec in enumerate(qr.Q.T):
    ax.arrow(0.0, 0.0, vec[0], vec[1], head_width=0.05, length_includes_head=True, linestyle="--", label=f"Q col {idx+1}")
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 2.5)
ax.set_title("Columns vs Orthonormal basis")
ax.legend()
ax.grid(True)
path = FIGURES_DIR / "qr_basis.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved QR basis figure to {path}")

fig, ax = plt.subplots(figsize=(5, 4))
cax = ax.imshow(qr.R, cmap="viridis", interpolation="nearest")
ax.set_title("Upper triangular R")
fig.colorbar(cax, ax=ax)
path_r = FIGURES_DIR / "qr_R_matrix.png"
fig.savefig(path_r, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved QR R figure to {path_r}")


## Singular Value Decomposition

SVD expresses A as U Σ V^T. Singular values stretch the axes of the unit circle, revealing dominant directions.


In [ ]:

matrix = np.array([[2.0, 1.0], [1.0, 3.0]], dtype=np.float64)
U, sigma, VT = np.linalg.svd(matrix)
print("Singular values:", sigma)

theta = np.linspace(0.0, 2 * np.pi, 200)
circle = np.column_stack([np.cos(theta), np.sin(theta)])
unit_axes = np.column_stack([np.array([1.0, 0.0]), np.array([0.0, 1.0])])
transformed = (matrix @ circle.T).T

def apply_svd(vecs: np.ndarray) -> np.ndarray:
    return (U * sigma) @ (VT @ vecs.T)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(circle[:, 0], circle[:, 1], linestyle=":", color="gray", label="unit circle")
ax.plot(transformed[:, 0], transformed[:, 1], color="tab:blue", label="A · circle")
for idx, vec in enumerate(unit_axes.T):
    stretched = (matrix @ vec)
    ax.arrow(0.0, 0.0, stretched[0], stretched[1], head_width=0.05, length_includes_head=True, label=f"A axis {idx+1}")
ax.set_title("Unit circle stretches along singular directions")
ax.set_aspect("equal", "box")
ax.grid(True)
ax.legend()
path_svd = FIGURES_DIR / "svd_circle.png"
fig.savefig(path_svd, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved SVD stretch figure to {path_svd}")


## Reconstruction via U Σ V^T

We verify that A equals U Σ V^T by comparing the product with the original matrix and plotting the residual norm per element.


In [ ]:

S = np.zeros_like(matrix)
np.fill_diagonal(S, sigma)
reconstruction = U @ S @ VT
residual = matrix - reconstruction
print("Reconstruction residual norm:", np.linalg.norm(residual))

fig, ax = plt.subplots(figsize=(5, 4))
cax = ax.imshow(residual, cmap="seismic", interpolation="nearest")
ax.set_title("Reconstruction residual")
fig.colorbar(cax, ax=ax)
path_res = FIGURES_DIR / "svd_residual.png"
fig.savefig(path_res, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved residual figure to {path_res}")
